<a href="https://colab.research.google.com/github/Patrick190508/mafia_island_tours/blob/main/notebooks/EDA_Global_Wind_Traffic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EDA — Sound proxies: wind and vessel traffic
**OBIS human sightings of *Rhincodon typus* (9,731 records)**

There is no global daily map of underwater noise, so two proxies are used:

| Proxy | Source | What it represents | Value per sighting |
|---|---|---|---|
| **Wind speed at 10 m** | Copernicus Marine (`cmems_obs-wind_glo_phy_my_l4_0.125deg_PT1H`, hourly → daily mean, 1999–) | natural ambient noise (waves, spray) | wind on the day and place of the sighting (m/s) |
| **Vessel traffic density** | Global Shipping Traffic Density raster (World Bank / IMF, mean 2015–2021) | anthropogenic noise (ships) | vessel hours in the cell of the sighting |

Question: in which wind and traffic conditions are whale sharks sighted?

Limitation: sightings depend on boat effort — strong wind also keeps boats in port, and traffic is a long-term average, not the value on the day.

## 1. Data

### 1.1 Libraries, sightings dataset and `map_builder`

In [1]:
!pip -q install copernicusmarine global-land-mask

import pandas as pd, numpy as np, xarray as xr, os
import matplotlib.pyplot as plt
import copernicusmarine
from global_land_mask import globe

URL = "https://raw.githubusercontent.com/Patrick190508/whaleshark-mafia-island/main/data/obis_whaleshark_clean_enriched_sites.csv"
db = pd.read_csv(URL, low_memory=False)
db["date"] = pd.to_datetime(db["date"]).dt.normalize()
db["year"] = db["date"].dt.year
db["month"] = db["date"].dt.month
print(db.shape)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.5/130.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.9/376.9 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 5.3 MB/s eta 0:00:00
(9731, 28)


Map Builder Function

In [2]:
def map_builder(subset, title, color, bbox=(-180, 180, -60, 60), step=None):
    lon0, lon1, lat0, lat1 = bbox
    if step is None:
        step = 0.5 if (lon1 - lon0) > 90 else 0.05
    lons = np.arange(lon0, lon1, step)
    lats = np.arange(lat0, lat1, step)
    LON, LAT = np.meshgrid(lons, lats)
    land = globe.is_land(LAT, LON)

    w = 14
    h = w * (lat1 - lat0) / (lon1 - lon0) + 1
    fig, ax = plt.subplots(figsize=(w, h))
    ax.contourf(LON, LAT, land, levels=[0.5, 1.5], colors=["#e8dcc8"])
    ax.scatter(db["decimalLongitude"], db["decimalLatitude"], s=3, c="lightgrey", label="all sightings")
    ax.scatter(subset["decimalLongitude"], subset["decimalLatitude"], s=6, c=color, label=title)
    ax.set_xlim(lon0, lon1); ax.set_ylim(lat0, lat1); ax.set_aspect("equal")
    ax.set_title(title); ax.legend(loc="lower left", fontsize=8)
    plt.show()

### 1.2 Wind at each sighting
Daily mean wind speed at the nearest grid point on the day of the sighting (`wind_speed`, m/s). Checkpoint file so the cell can be re-run after a disconnect.

In [3]:
copernicusmarine.login()

CKPT = "wind_progress.csv"
wind_ds = copernicusmarine.open_dataset(
    dataset_id="cmems_obs-wind_glo_phy_my_l4_0.125deg_PT1H",
    variables=["wind_speed"])
wind = wind_ds["wind_speed"]
t0 = str(wind.time.values[0])[:10]
print("wind data from", t0, "to", str(wind.time.values[-1])[:10])

if "wind_speed" not in db.columns:
    db["wind_speed"] = np.nan
if os.path.exists(CKPT):
    prog = pd.read_csv(CKPT, index_col=0)["wind_speed"]
    db["wind_speed"] = db["wind_speed"].fillna(db["id"].map(prog))

todo = db[db["wind_speed"].isna() & (db["date"] >= t0)]
days = sorted(todo["date"].unique())
print("days to do:", len(days), "| records:", len(todo))

for k, day in enumerate(days, 1):
    rows = todo[todo["date"] == day]
    pts = wind.sel(time=slice(day, day + pd.Timedelta(hours=23))).sel(
        latitude=xr.DataArray(rows["decimalLatitude"].values, dims="p"),
        longitude=xr.DataArray(rows["decimalLongitude"].values, dims="p"),
        method="nearest")
    db.loc[rows.index, "wind_speed"] = pts.mean("time").values
    if k % 100 == 0 or k == len(days):
        db[["id", "wind_speed"]].dropna().set_index("id").to_csv(CKPT)
        print(f"{k}/{len(days)} days done")

print("coverage:", f"{db['wind_speed'].notna().mean():.0%}")

OUT = "obis_whaleshark_sound_proxies.csv"
db.to_csv(OUT, index=False)
from google.colab import files
files.download(OUT)

INFO - 2026-09-22T16:00:33Z - Downloading Copernicus Marine data requires a Copernicus Marine username and password, sign up for free at: https://data.marine.copernicus.eu/register
INFO:copernicusmarine:Downloading Copernicus Marine data requires a Copernicus Marine username and password, sign up for free at: https://data.marine.copernicus.eu/register


Copernicus Marine username: psilingardi
Copernicus Marine password: ··········


INFO - 2026-09-22T16:00:42Z - Credentials file stored in /root/.copernicusmarine/.copernicusmarine-credentials.
INFO:copernicusmarine:Credentials file stored in /root/.copernicusmarine/.copernicusmarine-credentials.
INFO - 2026-09-22T16:00:44Z - Selected dataset version: "202211"
INFO:copernicusmarine:Selected dataset version: "202211"
INFO - 2026-09-22T16:00:44Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"


VariableDoesNotExistInTheDataset: The variable 'wind_speed' is neither a variable or a standard name in the dataset.

### 1.3 Vessel traffic at each sighting
Value of the traffic-density raster at the sighting position (`traffic`, vessel hours per cell).

### 1.4 Coverage check and save
Share of records with wind / traffic; save `obis_whaleshark_sound_proxies.csv` to GitHub `data/`.

## 2. Wind

### 2.1 Distribution of wind speed on sighting days

*Reading:*

### 2.2 World map binned by wind class
Classes: 0–3, 3–5, 5–7, 7–10, >10 m/s.

*Reading:*

## 3. Vessel traffic

### 3.1 Distribution of traffic density at sighting positions

*Reading:*

### 3.2 World map binned by traffic class
Classes from the data (quantiles).

*Reading:*

## 4. Summary and conclusions

| Proxy | Where sightings fall | Limitation |
|---|---|---|
| Wind | | boats stay in port on windy days |
| Vessel traffic | | long-term average, not the day of the sighting |

*Conclusions:*